# Wan 2.2 Image-to-Video — Official Repository

Runs the **official [`Wan-Video/Wan2.2`](https://github.com/Wan-Video/Wan2.2)** `generate.py` (task `ti2v-5B`), pinned to a specific upstream commit. No `diffusers` pipeline, no `AutoPipelineForImage2Video`.

**To run it:** `Runtime -> Run all` (or `Restart session and run all`). There is one code cell. It clones the repo, installs dependencies, downloads the model, pauses **once** for you to upload a product photo, generates a 9:16 MP4, and downloads it. Nothing else needs a click.

## Read this first: memory requirements

The official `ti2v-5B` pipeline builds its 10.6 GB text encoder and the 5B DiT in **system RAM** before any weights reach the GPU (`wan/textimage2video.py`), so it needs roughly **20+ GiB of RAM**, on top of the GPU's VRAM. The offload flags this notebook uses (`--offload_model True --convert_model_dtype --t5_cpu`) reduce *GPU* memory only.

- **Free Colab (about 12.7 GiB RAM) cannot run this pipeline.** The cell checks this first, before any download, and stops with a clear message rather than failing 30 minutes in.
- Choose a runtime with more RAM: a **High-RAM** option, or an **L4 / A100** GPU (`Runtime -> Change runtime type`). The notebook's metadata requests a T4 with High-RAM; Colab only applies that if your account has access to it.
- On a 16 GB T4, GPU memory can also run short (the official README lists 24 GB for this model). If it does, the cell automatically retries with a shorter clip (5s → 3s → 2s → 1s).
- The model download is about **32 GB**.

## How it avoids restart problems

Every heavy step (pip, the import check, the model download, `generate.py`) runs in its **own fresh child process**; the notebook's own kernel never imports `torch`, `numpy` or `diffusers`. So nothing is loaded in memory that a package install could leave stale, and no runtime restart is ever required.

## Python 3.13 and the official requirements

- `requirements.txt` is installed in three tiers: exact pins first; then with `numpy` unbounded (`numpy<2` has no Python 3.13 wheels); then fully unpinned.
- Wan's `wan/__init__.py` eagerly imports its Speech-to-Video and Animate pipelines, so `import wan` needs `decord`, `peft`, `librosa` and `einops` even though `ti2v-5B` never uses the first three. **`requirements.txt` doesn't list any of them.** They're installed explicitly; if one still can't be installed, a harmless stand-in is created for it (never for `einops`, which the `ti2v-5B` VAE genuinely needs).
- `torch`, `torchvision` and `torchaudio` stay as Colab's own GPU-matched build. `flash_attn` is skipped: its imports in Wan are guarded and the attention code falls back to PyTorch's `scaled_dot_product_attention`.

## Output size

`704*1280` is `ti2v-5B`'s only 9:16 option. `480*832` is not valid for this task (`wan/configs/__init__.py` restricts `ti2v-5B` to `704*1280` and `1280*704` and raises `AssertionError` otherwise).

In [ ]:
import os
import re
import shutil
import subprocess
import sys

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------
REPO_URL = 'https://github.com/Wan-Video/Wan2.2.git'
REPO_COMMIT = '42bf4cfaa384bc21833865abc2f9e6c0e67233dc'  # upstream main, 2026-03-17 (the code this notebook was checked against)
REPO_DIR = '/content/Wan2.2'
MODEL_REPO = 'Wan-AI/Wan2.2-TI2V-5B'
MODEL_DIR = './Wan2.2-TI2V-5B'
STUB_DIR = '/content/wan_stubs'

SIZE = '704*1280'                       # ti2v-5B's only 9:16 option (480*832 is not valid for this task)
FPS = 24                                # fixed by the ti2v-5B model config (sample_fps)
DURATION_LADDER_SECONDS = [5, 3, 2, 1]  # automatic fallback order if a length runs out of memory
PROMPT = 'A realistic, premium commercial product video. Natural, smooth camera motion, cinematic lighting.'  # edit me
SAVE_FILE = 'output.mp4'

# Wan's ti2v-5B pipeline builds its 10.6 GB text encoder and the 5B DiT in *system RAM* before any
# weights reach the GPU (see wan/textimage2video.py), so it needs roughly 20+ GiB of RAM no matter
# which GPU is attached. Set this to True only if you want to try a lower-RAM runtime anyway.
MIN_RAM_GIB = 20
FORCE_RUN_ON_LOW_RAM = False

# Every heavy step below (pip, the import check, the model download, generate.py) runs in its
# own fresh child process. This notebook's own kernel never imports torch/numpy/diffusers, so
# there is nothing loaded in memory that a package install could leave stale -- which is what
# removes the need for any runtime restart.
CHILD_ENV = dict(os.environ)
CHILD_ENV['PYTHONPATH'] = STUB_DIR + os.pathsep + CHILD_ENV.get('PYTHONPATH', '')
CHILD_ENV['PYTHONUNBUFFERED'] = '1'


def step(message):
    print(f'\n=== {message} ===')


def run_captured(cmd):
    return subprocess.run(cmd, capture_output=True, text=True, env=CHILD_ENV)


def run_streamed(cmd):
    """Run cmd, printing its output live. Returns (exit code, last ~8000 characters of output)."""
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=CHILD_ENV,
    )
    tail = ''
    for line in proc.stdout:
        print(line, end='')
        tail = (tail + line)[-8000:]
    proc.wait()
    return proc.returncode, tail


def pip_install(specs):
    return run_captured([sys.executable, '-m', 'pip', 'install', '-q'] + specs)


def show_error(result, chars=1200):
    text = ((result.stderr or '') + (result.stdout or '')).strip()
    print('    ' + text[-chars:].replace('\n', '\n    '))


# ---------------------------------------------------------------------------
# 1) Check the runtime: GPU and system RAM (fail fast, before any download)
# ---------------------------------------------------------------------------
step('1/8 Checking the runtime (GPU + system RAM)')


def total_ram_gib():
    with open('/proc/meminfo') as f:
        for line in f:
            if line.startswith('MemTotal:'):
                return int(line.split()[1]) / (1024 * 1024)  # kB -> GiB
    return None


try:
    smi = run_captured(['nvidia-smi'])
except FileNotFoundError:
    smi = None
if smi is None or smi.returncode != 0:
    raise RuntimeError(
        'No GPU detected. Click Runtime > Change runtime type, choose a GPU, Save, '
        'then run this cell again.'
    )
print(smi.stdout)

ram_gib = total_ram_gib()
ram_text = f'{ram_gib:.1f} GiB' if ram_gib else 'unknown'
print(f'System RAM: {ram_text}')
if ram_gib is not None and ram_gib < MIN_RAM_GIB and not FORCE_RUN_ON_LOW_RAM:
    raise RuntimeError(
        f'This runtime has only {ram_gib:.1f} GiB of system RAM. The official ti2v-5B pipeline loads '
        'its 10.6 GB text encoder and the 5B DiT into system RAM before moving weights to the GPU, '
        f'so it needs roughly {MIN_RAM_GIB}+ GiB; on less, the process is killed mid-load (after a '
        '~32 GB model download). The free Colab tier (about 12.7 GiB) is below that, so this cannot '
        'work there whatever the GPU setting. Choose a runtime with more RAM -- Runtime > Change '
        'runtime type > a High-RAM option, or an L4 / A100 GPU -- then run this cell again. '
        '(To try anyway, set FORCE_RUN_ON_LOW_RAM = True at the top of this cell.)'
    )

# ---------------------------------------------------------------------------
# 2) Clone the official repository (pinned to the commit this notebook was checked against)
# ---------------------------------------------------------------------------
step('2/8 Cloning the official Wan-Video/Wan2.2 repository')
os.chdir('/content')
shutil.rmtree(REPO_DIR, ignore_errors=True)
clone = run_captured(['git', 'clone', '--quiet', REPO_URL, REPO_DIR])
if clone.returncode != 0:
    show_error(clone)
    raise RuntimeError('git clone failed. Check your network connection and run this cell again.')
os.chdir(REPO_DIR)
checkout = run_captured(['git', 'checkout', '--quiet', REPO_COMMIT])
if checkout.returncode != 0:
    show_error(checkout)
    raise RuntimeError(f'Could not check out the pinned commit {REPO_COMMIT}.')
print(f'Repository ready at {REPO_DIR} (commit {REPO_COMMIT[:7]})')

# ---------------------------------------------------------------------------
# 3) Install dependencies
#    - requirements.txt, in three tiers, because some of its pins have no Python 3.13 wheel
#    - the packages `import wan` needs but requirements.txt never lists
# ---------------------------------------------------------------------------
step('3/8 Installing dependencies')

# torch/torchvision/torchaudio stay as Colab's own GPU-matched build (reinstalling them is a
# common way to break CUDA). flash_attn is optional: Wan's attention code falls back to
# PyTorch's scaled_dot_product_attention when it is missing.
EXCLUDE = ('flash_attn', 'torch', 'torchvision', 'torchaudio')
with open('requirements.txt') as f:
    lines = [line.strip() for line in f if line.strip() and not line.strip().startswith('#')]
specs = [s for s in lines if not s.lower().startswith(EXCLUDE)]


def unpin(spec):
    return re.sub(r'\s*[<>=!~].*$', '', spec)


tiers = [
    ('exact versions from requirements.txt', specs),
    ('same, with numpy unbounded (numpy<2 has no Python 3.13 wheels)',
     ['numpy' if unpin(s).lower() == 'numpy' else s for s in specs]),
    ('all packages unpinned', [unpin(s) for s in specs]),
]
installed = False
for label, tier_specs in tiers:
    print(f'  Trying: {label}')
    result = pip_install(tier_specs)
    if result.returncode == 0:
        print('    OK')
        installed = True
        break
    print('    Failed:')
    show_error(result, 700)
if not installed:
    raise RuntimeError('Dependency install failed on every attempt. See the pip errors above.')

# `import wan` (which generate.py runs first) eagerly imports Wan's Speech-to-Video and Animate
# pipelines, so these must be importable even though ti2v-5B never uses decord/librosa/peft.
for package in ['einops', 'peft', 'librosa', 'decord', 'huggingface_hub']:
    result = pip_install([package])
    if result.returncode != 0 and package == 'decord':
        result = pip_install(['eva-decord'])  # community fork with newer wheels, same import name
    print(f'  {package}: ' + ('OK' if result.returncode == 0 else 'not installable here (a stand-in is used if needed)'))

# ---------------------------------------------------------------------------
# 4) Verify the environment exactly the way generate.py will use it (fresh child process)
# ---------------------------------------------------------------------------
step('4/8 Verifying the environment')

STUBBABLE = ('decord', 'librosa', 'peft')  # imported by `import wan` but unused by ti2v-5B
STUB_SOURCE = '\n'.join([
    '# Stand-in generated by the Wan 2.2 Colab notebook.',
    '# wan/__init__.py eagerly imports Wan\'s Speech-to-Video and Animate pipelines, which need this',
    '# package at import time. The ti2v-5B flow never calls into it.',
    'def __getattr__(attr):',
    '    if attr.startswith(\'__\'):',
    '        raise AttributeError(attr)',
    '    def _unavailable(*args, **kwargs):',
    '        raise RuntimeError(attr + \' is a stand-in in this environment\')',
    '    return _unavailable',
    '',
])

CHECK_CODE = '\n'.join([
    'import torch',
    'print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())',
    'assert torch.cuda.is_available(), "CUDA is not available to PyTorch"',
    'import wan',
    'print("wan import: OK")',
])

verified = False
for _ in range(4):
    check = run_captured([sys.executable, '-c', CHECK_CODE])
    print(check.stdout, end='')
    if check.returncode == 0:
        verified = True
        break
    error_text = (check.stderr or '')[-3000:]
    if 'CUDA is not available to PyTorch' in error_text:
        raise RuntimeError(
            'PyTorch cannot see the GPU. Click Runtime > Change runtime type, choose T4 GPU, '
            'Save, then run this cell again.'
        )
    stub_name = next(
        (m for m in STUBBABLE
         if (f"No module named '{m}'" in error_text or f'/{m}/' in error_text)
         and not os.path.exists(os.path.join(STUB_DIR, m))),
        None,
    )
    if stub_name is None:
        print(error_text)
        raise RuntimeError('`import wan` failed after installing dependencies. See the error above.')
    print(f'  {stub_name} is unavailable here; adding a stand-in (ti2v-5B never calls into it) and re-checking...')
    os.makedirs(os.path.join(STUB_DIR, stub_name), exist_ok=True)
    with open(os.path.join(STUB_DIR, stub_name, '__init__.py'), 'w') as f:
        f.write(STUB_SOURCE)
if not verified:
    raise RuntimeError('`import wan` still fails after adding stand-ins. See the output above.')

# ---------------------------------------------------------------------------
# 5) Download the T4-compatible model (TI2V-5B)
# ---------------------------------------------------------------------------
step('5/8 Downloading ' + MODEL_REPO + ' (about 32 GB on first run -- this takes a while)')
download_code = (
    'from huggingface_hub import snapshot_download; '
    f"snapshot_download(repo_id='{MODEL_REPO}', local_dir='{MODEL_DIR}')"
)
code, tail = run_streamed([sys.executable, '-c', download_code])
if code != 0:
    raise RuntimeError('Model download failed. The output above shows why; run this cell again to resume.')
for required in ('models_t5_umt5-xxl-enc-bf16.pth', 'Wan2.2_VAE.pth'):
    if not os.path.exists(os.path.join(MODEL_DIR, required)):
        raise RuntimeError(f'Model download finished but {required} is missing; run this cell again to resume.')
print('Model ready.')

# ---------------------------------------------------------------------------
# 6) Upload your product image -- the only manual step
# ---------------------------------------------------------------------------
step('6/8 Waiting for you to upload one product photo')
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file was uploaded. Run this cell again and choose one product photo.')
image_filename = next(iter(uploaded))
image_path = os.path.join(REPO_DIR, image_filename)
with open(image_path, 'wb') as f:
    f.write(uploaded[image_filename])
print(f'Uploaded: {image_filename}')

# ---------------------------------------------------------------------------
# 7) Generate the 9:16 video with the official generate.py.
#    The officially documented low-memory flags are always on (--offload_model True,
#    --convert_model_dtype, --t5_cpu). If the GPU still runs out of memory, the clip is
#    shortened automatically (fresh process each time, so GPU memory is fully released).
# ---------------------------------------------------------------------------
step('7/8 Generating the video')


def frame_num_for(seconds):
    return int(round((FPS * seconds - 1) / 4)) * 4 + 1  # frame counts must be 4n+1


def run_generate(seconds):
    frame_num = frame_num_for(seconds)
    cmd = [
        sys.executable, 'generate.py',
        '--task', 'ti2v-5B',
        '--size', SIZE,
        '--ckpt_dir', MODEL_DIR,
        '--offload_model', 'True',
        '--convert_model_dtype',
        '--t5_cpu',
        '--image', image_path,
        '--prompt', PROMPT,
        '--frame_num', str(frame_num),
        '--save_file', SAVE_FILE,
    ]
    print(f'  Trying ~{frame_num / FPS:.1f}s ({frame_num} frames) at {SIZE}...')
    if os.path.exists(SAVE_FILE):
        os.remove(SAVE_FILE)
    return run_streamed(cmd)


index = 0
while index < len(DURATION_LADDER_SECONDS):
    seconds = DURATION_LADDER_SECONDS[index]
    code, tail = run_generate(seconds)
    if code == 0 and os.path.exists(SAVE_FILE) and os.path.getsize(SAVE_FILE) > 0:
        print(f'\nGenerated a ~{seconds}s clip: {SAVE_FILE}')
        break
    if 'out of memory' in tail.lower():
        print(f'  Out of memory at ~{seconds}s; automatically retrying with a shorter clip...')
        index += 1
        continue
    if code < 0:
        raise RuntimeError(
            f'generate.py was killed by the system (signal {-code}) -- almost certainly because system '
            f'RAM ran out ({ram_text} here). A shorter clip cannot help with that: the text '
            'encoder and the DiT are loaded into RAM before generation starts. Use a runtime with '
            'more RAM (a High-RAM option, or an L4 / A100 GPU) and run this cell again.'
        )
    raise RuntimeError(f'generate.py failed with exit code {code}. Its full output is above.')
else:
    raise RuntimeError(
        'Ran out of GPU memory even at the shortest fallback length. This session has less free '
        'memory than usual: Runtime > Disconnect and delete runtime, then run this notebook again '
        'to get a fresh machine, or use a GPU with more VRAM.'
    )

# ---------------------------------------------------------------------------
# 8) Preview and automatically download the MP4
# ---------------------------------------------------------------------------
step('8/8 Downloading the video')
from IPython.display import Video, display

display(Video(SAVE_FILE, embed=True))
files.download('output.mp4')
print('\nAll done -- output.mp4 has been downloaded.')